# Author reproduction and matched comparison — correction workspace

**No all-model reproduction/comparison is certified yet.** This notebook records the audit and gives a launcher for original scripts. It contains no fabricated model metrics and has not been executed on Kaggle. Original author experiments and the controlled DIV2K/Kodak benchmark are reported separately. Existing results/checkpoints remain historical artifacts.

| Model | Release | Required correction |
|---|---|---|
| DeepJSCC | third_party | Original-author 2019 code unverified; used repository is a third-party reproduction. |
| ADJSCC | author_training_available | Native ImageNet data/runtime and AWGN initialization/protocol must be reproduced; previous run is a DIV2K adaptation. |
| NTSCC | author_training_available | Separate 2022 model; required initialization, paths and 5000-epoch native-loop constraint unresolved. |
| NTSCC_plus | author_training_available | Restore original NATTEN-compatible environment and author training path; previous run used 128 images/epoch and added fading. |
| NTSCC++ | online_code_unreleased | Author README marks online NTSCC++ inference as TODO; cannot rename NTSCC+ to NTSCC++. |
| SwinJSCC | author_training_available | Base w/o-SA-and-RA architecture matches source settings; training/initialization and common evaluation still require runtime validation. |
| SQ-GAN | author_training_available | Previous analog channel/pseudo-label adaptation differs from the published codec; native Cityscapes reproduction and external digital transport study are distinct. |
| DeepJSCC-Q | candidate_source_found | Dedicated trainer found in r0ymanesco/deepjscc_stream; authorship and journal-variant fidelity not established. Old simplified notebook is not certified. |
| DiT-JSCC | author_inference_only | Public README explicitly omits training workspace; original training code/weights required. |
| DDM-SSCC | author_code_unverified | No author release verified; exact lossless codec/transport cannot be certified from a substitute model. |

## 1. Locate the workspace

On Kaggle, attach a private dataset containing the reproduction scripts, locked source manifests/snapshots and audit JSON files. Set SEMANTICLOSSLESS_ROOT to its mounted root. This notebook does not upload anything or allocate GPUs automatically.

In [ ]:
import os, json, pathlib, subprocess, sys
ROOT = pathlib.Path(os.environ.get('SEMANTICLOSSLESS_ROOT', '.'))
assert (ROOT / 'SharedCode/reproduction').is_dir(), 'Attach the real workspace files before continuing'
status = json.loads((ROOT / 'Results/reproduction_audit/experiment_eligibility.json').read_text())
print(json.dumps(status, indent=2))


## 2. Verify upstream source bytes

This checks saved source hashes, including dataset loaders. It does not prove that a notebook used an unchanged runtime or that training completed.

In [ ]:
import hashlib
locks = json.loads((ROOT / 'Results/reproduction_audit/source_lock.json').read_text())
for lock in locks:
    assert 'error' not in lock, lock
    source = ROOT / lock['source_directory']
    for name, digest in lock['files_sha256'].items():
        assert hashlib.sha256((source / name).read_bytes()).hexdigest() == digest, (lock['model'], name)
    print(lock['model'], lock['commit'], lock['file_count'], 'source bytes verified')


## 3. Native author training entry points

Use the original scripts and their own arguments in their required environments. Provide real datasets and any required author initialization checkpoints. The example below is SQ-GAN's original Cityscapes semantic-stage command. It does not use the previous analog channel or pseudo-label wrapper. The image stage and joint stage follow the original README and require their proper initialization/split components. The example has not been run here; dependency/data availability is not assumed.

In [ ]:
AUTHOR_PYTHON = os.environ.get('AUTHOR_PYTHON', sys.executable)
RUN_ROOT = pathlib.Path('/kaggle/working/author_reproduction') if pathlib.Path('/kaggle').exists() else ROOT / 'Results/author_reproduction'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
arguments_path = RUN_ROOT / 'sqgan_ssm_args.json'
arguments_path.write_text(json.dumps(['--mode','ssm','--gpus','-1','--base','config/sqgan_cityscapes.yml','--max_epochs','150']))
CITYSCAPES_ROOT = os.environ.get('CITYSCAPES_ROOT')
assert CITYSCAPES_ROOT and pathlib.Path(CITYSCAPES_ROOT).is_dir(), 'Original Cityscapes RGB and gtFine train-ID masks are required'
links_path = RUN_ROOT / 'sqgan_dataset_links.json'
links_path.write_text(json.dumps({'path/to/cityscapes': CITYSCAPES_ROOT}))
command = [sys.executable, str(ROOT / 'SharedCode/reproduction/launch_upstream.py'), '--model','SQ-GAN', '--entrypoint','train.py', '--python',AUTHOR_PYTHON, '--execution-dir',str(RUN_ROOT / 'SQGAN_native_ssm'), '--arguments-json',str(arguments_path), '--dataset-links-json',str(links_path)]
print(command)
# Prepare and verify the execution copy; no training in this cell.
subprocess.run(command, check=True)


In [ ]:
# Run this cell only in the original compatible GPU environment with verified inputs.
# This executes the real original trainer; all errors propagate.
# subprocess.run(command + ['--execute'], check=True)


## 4. Controlled DIV2K/Kodak evaluation

Use matched_evaluation.py after each author-code model callback and transport have been validated. Every Kodak pixel is reconstructed; metrics use full original images. All padded tiles count toward FLOPs and channel uses. Do not pass the old SQ-GAN analog vector adapter as the original discrete-index system. Author-native fading assumptions differ, and unsupported native channels cannot be silently replaced. The comparison validator rejects incomplete or incompatible run manifests. No callbacks/results are invented here.

In [ ]:
# Empty until real measured, reviewed runs exist. No fallback to historical scores.
eligible_run_manifests = []
if len(eligible_run_manifests) < 2:
    print('No certified matched ranking available yet.')
else:
    sys.path.insert(0, str(ROOT / 'SharedCode/reproduction'))
    from matched_evaluation import validate_comparison
    print(validate_comparison(eligible_run_manifests))
